In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

In [5]:
class SEBlock(nn.Module):
    def __init__(self, in_channels, reduction=8):
        super().__init__()
        self.fc1 = nn.Linear(in_channels, in_channels // reduction)
        self.fc2 = nn.Linear(in_channels // reduction, in_channels)

    def forward(self, x):
        se = x.mean(-1)
        se = F.relu(self.fc1(se))
        se = torch.sigmoid(self.fc2(se))
        se = se.unsqueeze(-1)
        return x * se

In [6]:
class ConvBlock(nn.Module):

    def __init__(self, in_ch, out_ch, kernel_size=7, stride=1, pool_k=4, dilation=1):

        super().__init__()

        padding = ((kernel_size - 1) // 2) * dilation

        self.conv1 = nn.Conv1d(in_ch, out_ch, kernel_size, stride, padding, dilation=dilation)
        self.bn1 = nn.BatchNorm1d(out_ch)

        self.conv2 = nn.Conv1d(out_ch, out_ch, kernel_size, 1, padding, dilation=dilation)
        self.bn2 = nn.BatchNorm1d(out_ch)

        self.pool = nn.MaxPool1d(pool_k) if pool_k else None

        self.se = SEBlock(out_ch)

    def forward(self, x):

        x = F.relu(self.bn1(self.conv1(x)))
        x = F.relu(self.bn2(self.conv2(x)))

        if self.pool:
            x = self.pool(x)

        x = self.se(x)

        return x

In [7]:
class SleepCNN(nn.Module):

    def __init__(self, n_channels=2, num_classes=5):

        super().__init__()

        self.block1 = ConvBlock(n_channels, 64, 7, 1, 4)
        self.block2 = ConvBlock(64, 128, 5, 1, 4)
        self.block3 = ConvBlock(128, 256, 3, 1, None, dilation=2)

        self.global_pool = nn.AdaptiveAvgPool1d(1)

        self.classifier = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(256,128),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(128,num_classes)
        )

    def forward(self,x):

        x=self.block1(x)
        x=self.block2(x)
        x=self.block3(x)

        x=self.global_pool(x)

        x=x.view(x.size(0),-1)

        return self.classifier(x)

In [11]:
model = SleepCNN(n_channels=2, num_classes=5)

checkpoint = torch.load("/kaggle/input/models/mok18976/eeg-final/pytorch/default/1/final_model.pth", map_location="cpu")

model.load_state_dict(checkpoint)

model.eval()

print("Model loaded successfully")

Model loaded successfully


In [18]:
#------------------------------------
#single data check
#-------------------------------------

X = np.load("/kaggle/input/datasets/mok18976/eeg-sleep-recognition-dataset/processed_data/SC4001E0_X.npy")
y = np.load("/kaggle/input/datasets/mok18976/eeg-sleep-recognition-dataset/processed_data/SC4001E0_y.npy")

print("Data shape:", X.shape)

epoch = X[0].astype(np.float32)

mean = epoch.mean(axis=-1, keepdims=True)
std = epoch.std(axis=-1, keepdims=True)

epoch = (epoch - mean) / (std + 1e-8)


input_tensor = torch.from_numpy(epoch).unsqueeze(0)

with torch.no_grad():

    logits = model(input_tensor)

    pred = torch.argmax(logits, dim=1)

print("Predicted stage:", pred.item())
print("True stage:", y[0])

Data shape: (2650, 2, 3000)
Predicted stage: 0
True stage: 0


In [16]:
import os
import glob

path = "/kaggle/input/datasets/mok18976/eeg-sleep-recognition-dataset/processed_data/"
x_files = sorted(glob.glob(path + "*_X.npy"))
y_files = sorted(glob.glob(path + "*_y.npy"))

print(f"Found {len(x_files)} files:")
for f in x_files:
    print(f)

Found 153 files:
/kaggle/input/datasets/mok18976/eeg-sleep-recognition-dataset/processed_data/SC4001E0_X.npy
/kaggle/input/datasets/mok18976/eeg-sleep-recognition-dataset/processed_data/SC4002E0_X.npy
/kaggle/input/datasets/mok18976/eeg-sleep-recognition-dataset/processed_data/SC4011E0_X.npy
/kaggle/input/datasets/mok18976/eeg-sleep-recognition-dataset/processed_data/SC4012E0_X.npy
/kaggle/input/datasets/mok18976/eeg-sleep-recognition-dataset/processed_data/SC4021E0_X.npy
/kaggle/input/datasets/mok18976/eeg-sleep-recognition-dataset/processed_data/SC4022E0_X.npy
/kaggle/input/datasets/mok18976/eeg-sleep-recognition-dataset/processed_data/SC4031E0_X.npy
/kaggle/input/datasets/mok18976/eeg-sleep-recognition-dataset/processed_data/SC4032E0_X.npy
/kaggle/input/datasets/mok18976/eeg-sleep-recognition-dataset/processed_data/SC4041E0_X.npy
/kaggle/input/datasets/mok18976/eeg-sleep-recognition-dataset/processed_data/SC4042E0_X.npy
/kaggle/input/datasets/mok18976/eeg-sleep-recognition-dataset/p

In [17]:
X_all = np.concatenate([np.load(f) for f in x_files], axis=0)
y_all = np.concatenate([np.load(f) for f in y_files], axis=0)

print(f"Total data shape: {X_all.shape}")
print(f"Total labels: {y_all.shape}")

Total data shape: (414961, 2, 3000)
Total labels: (414961,)


In [19]:
indices = np.random.choice(len(X_all), 20, replace=False)

correct = 0
results = []

for i, idx in enumerate(indices):
    epoch = X_all[idx].astype(np.float32)
    mean = epoch.mean(axis=-1, keepdims=True)
    std = epoch.std(axis=-1, keepdims=True)
    epoch = (epoch - mean) / (std + 1e-8)

    input_tensor = torch.from_numpy(epoch).unsqueeze(0)

    with torch.no_grad():
        logits = model(input_tensor)
        pred = torch.argmax(logits, dim=1).item()

    true = y_all[idx]
    is_correct = pred == true
    if is_correct:
        correct += 1
    results.append((i+1, idx, pred, true, "✅" if is_correct else "❌"))

print(f"{'#':<4} {'Index':<8} {'Predicted':<12} {'True':<8} {'Result'}")
print("-" * 45)
for r in results:
    print(f"{r[0]:<4} {r[1]:<8} {r[2]:<12} {r[3]:<8} {r[4]}")
print("-" * 45)
print(f"Accuracy: {correct}/20 ({correct/20*100:.1f}%)")

#    Index    Predicted    True     Result
---------------------------------------------
1    129512   0            0        ✅
2    240160   0            0        ✅
3    81028    0            0        ✅
4    392950   4            2        ❌
5    222454   0            0        ✅
6    162666   2            2        ✅
7    5194     0            0        ✅
8    287891   0            0        ✅
9    20624    2            2        ✅
10   195483   0            0        ✅
11   94374    0            0        ✅
12   138026   2            3        ❌
13   157556   0            0        ✅
14   100306   0            0        ✅
15   285396   1            1        ✅
16   128071   0            0        ✅
17   343355   0            0        ✅
18   93688    0            0        ✅
19   330233   0            0        ✅
20   4160     4            4        ✅
---------------------------------------------
Accuracy: 18/20 (90.0%)
